# Fase 4 · KPIs y datos para el dashboard
**Responsabilidad:** calcular los 6 indicadores clave y preparar las tablas que consume Tableau.
No vuelve a limpiar el archivo original ni cambia la población analítica (`incluido_analisis = True`).

| | |
|---|---|
| **Entrada** | `salida_fase3/airbnb_bogota_analitico.csv`, `salida_fase3/anfitriones_resumen.csv` |
| **Salida** | `resultados/kpis.csv`, `resultados/anuncios_dashboard.csv`, `resultados/anfitriones_dashboard.csv` |
| **Anterior / siguiente** | `01_preparacion_datos` → `03_contrastes_hipotesis` |

**Organización:** 04 verifica las exportaciones y documenta el cierre; las visualizaciones se mantienen en Tableau. No se exportan imágenes.

**Referencias del curso:** semana 4, pp. 1–3 (descriptivos), y semanas 7–8, pp. 4–6 (KPIs, segmentación y dashboard).


## 1. Carga

In [1]:
import numpy as np
import pandas as pd

from bi_airbnb import DIR_RESULTADOS, anfitriones_unicos, cargar_base, guardar_csv

todo, d, hosts = cargar_base()
print(f"Anuncios analizados: {len(d):,} de {len(todo):,} | anfitriones: {len(hosts):,}")

Anuncios analizados: 17,998 de 19,505 | anfitriones: 7,795


### Resumen descriptivo · Semana 4
Antes de los KPIs, revisamos tendencia central y dispersión de precio, capacidad y calificación. Cada columna usa sus datos no nulos. La varianza y desviación estándar son muestrales (`ddof=1`); el RIQ es Q3 − Q1. Comparar media y mediana ayuda a describir asimetría, sin convertirlo en una prueba formal. El precio está en COP, la varianza del precio en COP² y la calificación en puntos.


In [2]:
variables = d[["precio_cop", "capacidad", "calificacion"]]
descriptivos = variables.agg(["count", "mean", "median", "var", "std"]).T
descriptivos.columns = ["n", "media", "mediana", "varianza", "desviacion_estandar"]
descriptivos["q1"] = variables.quantile(.25)
descriptivos["q3"] = variables.quantile(.75)
descriptivos["riq"] = descriptivos.q3 - descriptivos.q1
descriptivos


,n,media,mediana,varianza,desviacion_estandar,q1,q3,riq
precio_cop,17998.0,183293.755852,160000.00,1.181857e+10,108713.244802,110000.00,221285.7675,111285.7675
capacidad,17998.0,3.049728,2.00,3.614519e+00,1.901189,2.00,4.0000,2.0000
calificacion,14549.0,4.704137,4.83,2.194958e-01,0.468504,4.65,4.9700,0.3200


## 2. KPIs
Cada fila lleva su `n_validos`, la hipótesis que alimenta y la fórmula equivalente en Tableau.

La guía de semanas 7–8 propone al menos tres KPIs: aquí se mantienen seis con denominador y propósito. La mediana resume tarifas asimétricas; la media describe la calificación disponible. La actividad por reseñas no equivale a reservas verificadas.


In [3]:
host_count = anfitriones_unicos(d)
rated = d.calificacion.notna().sum()

kpis = pd.DataFrame([
    ["K1", "Precio mediano por noche", d.precio_cop.median(), "COP", len(d), "P1 y P2", "MEDIAN([precio_cop])"],
    ["K2", "Precio mediano por huésped", d.precio_por_huesped.median(), "COP", d.precio_por_huesped.count(), "H3", "MEDIAN([precio_por_huesped])"],
    ["K3", "Calificación promedio", d.calificacion.mean(), "puntos sobre 5", rated, "H4 y P5", "AVG([calificacion])"],
    ["K4", "Anuncios con reseñas en 12 meses", d.resenas_12m.gt(0).mean(), "fracción", d.resenas_12m.count(), "P5", "SUM(IIF([resenas_12m]>0,1,0))/COUNT([resenas_12m])"],
    ["K5", "Ocupación estimada mediana", d.ocupacion_estimada_pct.median(), "fracción", d.ocupacion_estimada_pct.count(), "P5", "MEDIAN([ocupacion_estimada_pct])"],
    ["K6", "Anfitriones con varios anuncios", host_count.n_anuncios.ge(2).mean(), "fracción", len(host_count), "P5", "COUNTD(IF [n_anuncios]>=2 THEN [id_host] END)/COUNTD([id_host])"],
], columns=["id", "indicador", "valor", "unidad", "n_validos", "hipotesis", "formula_tableau"])
guardar_csv(kpis, DIR_RESULTADOS, "kpis.csv")
kpis[["id", "indicador", "valor", "n_validos"]]

,id,indicador,valor,n_validos
0,K1,Precio mediano por noche,160000.000000,17998
1,K2,Precio mediano por huésped,60000.000000,17998
2,K3,Calificación promedio,4.704137,14549
3,K4,Anuncios con reseñas en 12 meses,0.671797,17998
4,K5,Ocupación estimada mediana,0.065753,17998
5,K6,Anfitriones con varios anuncios,0.305067,7795


## 3. Tablas para Tableau
Los prefijos `a_` y `h_` garantizan que Tableau importe las claves como **texto**.

In [4]:
anuncios = d.copy()
anuncios["id_anuncio"] = "a_" + anuncios["id_anuncio"]
anuncios["id_host"] = "h_" + anuncios["id_host"]
anuncios["precio_intervalo_50mil"] = (np.floor(anuncios["precio_cop"] / 50000) * 50000).astype(int)
anuncios["activo_12m"] = anuncios["resenas_12m"].gt(0).astype(int)
anuncios["grupo_anfitrion"] = np.where(anuncios["n_anuncios"] >= 2, "Varios anuncios", "Un anuncio")
guardar_csv(anuncios, DIR_RESULTADOS, "anuncios_dashboard.csv")

anfitriones = hosts.copy()
anfitriones["id_host"] = "h_" + anfitriones["id_host"]
guardar_csv(anfitriones, DIR_RESULTADOS, "anfitriones_dashboard.csv")
print(f"anuncios_dashboard: {anuncios.shape} | anfitriones_dashboard: {anfitriones.shape}")

anuncios_dashboard: (17998, 51) | anfitriones_dashboard: (7795, 9)
